# PixelWarrior Project: Canonical Baseline Run on Kaggle (2x NVIDIA T4 DDP)

**Objective:** Establish the canonical baseline using both Kaggle NVIDIA T4 GPUs via PyTorch DistributedDataParallel (DDP) with exact batch size fidelity.

### Multi-GPU Batch Configuration:
- **Hardware:** $2 \times$ NVIDIA Tesla T4 GPUs (16 GB VRAM each)
- **World Size:** 2
- **Per-GPU Batch Size:** 2
- **Effective Global Batch Size:** $2 \times 2 = 4$ (**Exact match to audited baseline**)
- **Learning Rate:** $1 \times 10^{-4}$ ($0.0001$, unchanged)
- **Optimizer:** Adam (default $\beta_1=0.9, \beta_2=0.999$)
- **Scheduler:** CosineAnnealingLR ($T_{\text{max}}=15, \eta_{\text{min}}=10^{-6}$)
- **Loss:** MSELoss
- **Dataset:** 485 paired images ($600 \times 400$ native resolution)
- **Split:** Persisted 435 train / 50 val split (Seed 42)
- **Epochs:** 15 epochs (Fresh start from Epoch 0)
- **Outputs:** `/kaggle/working/baseline/`

In [ ]:
# 1. Multi-GPU Verification (Verify 2x T4 GPUs)
import torch
import os, sys

gpu_count = torch.cuda.device_count()
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Visible GPU Count: {gpu_count}")

if gpu_count < 2:
    print("\n[!] CAUTION: Less than 2 GPUs detected!")
    print("    Please select 'GPU T4 x 2' under Notebook Settings -> Accelerator in the right sidebar.")
else:
    print("\n[+] Confirmed 2x GPUs ready for DistributedDataParallel:")
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"    GPU [{i}]: {props.name} ({props.total_memory / (1024**3):.2f} GB VRAM)")

In [ ]:
# 2. Run Pre-Training Smoke Test
# Verifies DDP DistributedSampler partitioning, split integrity, and metric range clamping
!python tests/test_ddp_and_split.py

In [ ]:
# 3. Launch Canonical Baseline on 2x T4 GPUs via torchrun
# torchrun automatically coordinates 2 processes across cuda:0 and cuda:1
!torchrun --nproc_per_node=2 kaggle_train.py \
    --epochs 15 \
    --per_gpu_batch_size 2 \
    --lr 0.0001 \
    --seed 42 \
    --output_dir /kaggle/working/baseline

In [ ]:
# 4. Display Training History & Convergence Metrics
import json
import pandas as pd
import matplotlib.pyplot as plt

history_file = '/kaggle/working/baseline/train_history.json'
if os.path.isfile(history_file):
    with open(history_file, 'r') as f:
        hist = json.load(f)
    
    epochs_df = pd.DataFrame(hist['epochs'])
    display_cols = ['epoch', 'train_loss', 'train_psnr', 'val_mse', 'val_psnr', 'val_ssim', 'val_psnr_min', 'val_psnr_max', 'duration_sec']
    print("\n" + "="*60)
    print("CANONICAL BASELINE METRICS SUMMARY:")
    print(epochs_df[display_cols].to_string(index=False))
    print("="*60)
    print(f"Best Validation PSNR: {hist['best_val_psnr']:.2f} dB")
    print(f"Total Training Time:  {hist.get('total_time_min', 0):.2f} minutes")
    
    # Plot PSNR and SSIM curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs_df['epoch'], epochs_df['train_psnr'], label='Train PSNR (dB)', marker='o')
    ax1.plot(epochs_df['epoch'], epochs_df['val_psnr'], label='Val PSNR (dB)', marker='s', color='orange')
    ax1.set_title('PSNR Progression (dB)')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('PSNR (dB)')
    ax1.grid(True)
    ax1.legend()
    
    ax2.plot(epochs_df['epoch'], epochs_df['val_ssim'], label='Val SSIM', marker='^', color='green')
    ax2.set_title('Validation SSIM Progression')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('SSIM')
    ax2.grid(True)
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/baseline/convergence_curves.png')
    plt.show()
else:
    print(f"History file not found at {history_file}")

In [ ]:
# 5. Check Output Checkpoints
!ls -lh /kaggle/working/baseline/checkpoints